# 06 — Đánh giá & Đối sánh mô hình (Model Evaluation)

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 6** trong quy trình XAI-Football:
- Đánh giá toàn diện và đối sánh công bằng giữa các mô hình GNN (Tầng 2: GCN, MPNN, GATv2, GraphSAGE) và Baseline phi đồ thị (Tầng 3: Logistic Regression, Random Forest, ANN).
- Các độ đo: AUC-PR (chỉ số chính do mất cân bằng nhãn), AUC-ROC, F1, Precision, Recall, Brier score (đo độ calibration xác suất).
- Vẽ đường cong Precision-Recall (PR) và ROC curve từ dự đoán thực tế trên tập Test.
- Trực quan hóa Brier score phản ánh mức độ tin cậy của xác suất.
- Diễn giải kết quả thực tế một cách trung thực, khoa học và minh bạch.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 6 — Đánh giá mô hình (`scripts/07_evaluate.py`, `scripts/06_train_baselines.py`, `src/xai_football/evaluation/`)**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import auc, precision_recall_curve, roc_curve

REPO_ROOT = Path(".").resolve()
figures_dir = REPO_ROOT / "results" / "figures" / "official"
figures_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 150

## 3. Input và nguồn dữ liệu
- Bảng đối sánh tổng hợp: `results/tables/official/model_comparison.csv`
- Bảng đối sánh baseline: `results/tables/official/baseline_comparison.csv`
- File xác suất dự đoán tập test của baseline: `experiments/official/baselines/test_predictions.parquet`
- File xác suất dự đoán tập test của MPNN: `experiments/official/mpnn/test_predictions.parquet`

## 4. Config/script/module sử dụng
- Scripts: `scripts/07_evaluate.py`, `scripts/06_train_baselines.py`
- Module: `src/xai_football/evaluation/metrics.py`.

## 5. Các bước xử lý & Đọc kết quả thực tế

In [ ]:
model_comp_path = REPO_ROOT / "results" / "tables" / "official" / "model_comparison.csv"
if not model_comp_path.exists():
    raise FileNotFoundError(f"Thiếu file: {model_comp_path}. Vui lòng chạy evaluate pipeline trước.")

df_models = pd.read_csv(model_comp_path)
print("=== BẢNG KẾT QUẢ ĐỐI SÁNH CÁC MÔ HÌNH TRÊN TẬP TEST ===")
display(df_models)

## 6. Kết quả và cách diễn giải
### 6.1. Đối sánh AUC-PR & AUC-ROC giữa các mô hình

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sắp xếp theo AUC-PR
df_sorted = df_models.sort_values("auc_pr", ascending=True)
colors = ["#2980b9" if "GNN" in str(t) else "#e67e22" for t in df_sorted["tier"]]

# Biểu đồ AUC-PR
bars1 = axes[0].barh(df_sorted["model"], df_sorted["auc_pr"], color=colors, height=0.6)
axes[0].set_title("Đối sánh AUC-PR trên tập Test (Độ đo chính)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("AUC-PR", fontsize=10)
axes[0].set_xlim(0, 0.75)
for b in bars1:
    axes[0].text(b.get_width() + 0.01, b.get_y() + b.get_height() / 2, f"{b.get_width():.4f}", va="center", fontweight="bold", fontsize=9)

# Biểu đồ AUC-ROC
bars2 = axes[1].barh(df_sorted["model"], df_sorted["auc_roc"], color=colors, height=0.6)
axes[1].set_title("Đối sánh AUC-ROC trên tập Test", fontsize=12, fontweight="bold")
axes[1].set_xlabel("AUC-ROC", fontsize=10)
axes[1].set_xlim(0, 1.0)
for b in bars2:
    axes[1].text(b.get_width() + 0.01, b.get_y() + b.get_height() / 2, f"{b.get_width():.4f}", va="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.savefig(figures_dir / "06_model_auc_comparison.png", dpi=300)
plt.savefig(figures_dir / "06_model_auc_comparison.svg")
plt.show()

### 6.2. Đường cong Precision-Recall và ROC Curves từ Prediction Scores thật

In [ ]:
# Đọc file test_predictions thực tế từ baseline và MPNN
pred_bl_path = REPO_ROOT / "experiments" / "official" / "baselines" / "test_predictions.parquet"
pred_mpnn_path = REPO_ROOT / "experiments" / "official" / "mpnn" / "test_predictions.parquet"

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if pred_bl_path.exists():
    df_bl_preds = pd.read_parquet(pred_bl_path)
    y_test = df_bl_preds["y_test"].to_numpy()

    # Đường cong cho Random Forest
    rf_p, rf_r, _ = precision_recall_curve(y_test, df_bl_preds["rf_prob"])
    rf_fpr, rf_tpr, _ = roc_curve(y_test, df_bl_preds["rf_prob"])
    axes[0].plot(rf_r, rf_p, label=f"Random Forest (AUC-PR = {auc(rf_r, rf_p):.4f})", color="#27ae60", lw=2)
    axes[1].plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC-ROC = {auc(rf_fpr, rf_tpr):.4f})", color="#27ae60", lw=2)

    # Đường cong cho ANN
    ann_p, ann_r, _ = precision_recall_curve(y_test, df_bl_preds["ann_prob"])
    ann_fpr, ann_tpr, _ = roc_curve(y_test, df_bl_preds["ann_prob"])
    axes[0].plot(ann_r, ann_p, label=f"ANN MLP (AUC-PR = {auc(ann_r, ann_p):.4f})", color="#e67e22", lw=2)
    axes[1].plot(ann_fpr, ann_tpr, label=f"ANN MLP (AUC-ROC = {auc(ann_fpr, ann_tpr):.4f})", color="#e67e22", lw=2)

if pred_mpnn_path.exists():
    df_mp_preds = pd.read_parquet(pred_mpnn_path)
    y_mp = df_mp_preds["y_true"].to_numpy()
    mp_p, mp_r, _ = precision_recall_curve(y_mp, df_mp_preds["y_prob"])
    mp_fpr, mp_tpr, _ = roc_curve(y_mp, df_mp_preds["y_prob"])
    axes[0].plot(mp_r, mp_p, label=f"MPNN (AUC-PR = {auc(mp_r, mp_p):.4f})", color="#e74c3c", lw=2.5)
    axes[1].plot(mp_fpr, mp_tpr, label=f"MPNN (AUC-ROC = {auc(mp_fpr, mp_tpr):.4f})", color="#e74c3c", lw=2.5)

# PR Curve settings
axes[0].set_title("Đường cong Precision-Recall (Test Set)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Recall", fontsize=10)
axes[0].set_ylabel("Precision", fontsize=10)
axes[0].legend(loc="upper right")

# ROC Curve settings
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Đường cong ROC (Test Set)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("False Positive Rate", fontsize=10)
axes[1].set_ylabel("True Positive Rate", fontsize=10)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig(figures_dir / "06_pr_roc_curves.png", dpi=300)
plt.savefig(figures_dir / "06_pr_roc_curves.svg")
plt.show()

### 6.3. Brier Score — Độ tin cậy và Calibration của xác suất

In [ ]:
# Brier score càng thấp, mô hình dự đoán xác suất càng chuẩn xác
df_brier = df_models.dropna(subset=["brier_score"]).sort_values("brier_score", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#27ae60" if b < 0.12 else "#f39c12" if b < 0.20 else "#e74c3c" for b in df_brier["brier_score"]]
bars = ax.barh(df_brier["model"], df_brier["brier_score"], color=colors, height=0.55)
ax.set_title("Độ đo Brier Score trên tập Test (Càng thấp càng tốt)", fontsize=12, fontweight="bold")
ax.set_xlabel("Brier Score", fontsize=10)
ax.set_xlim(0, 0.28)

for b in bars:
    ax.text(b.get_width() + 0.005, b.get_y() + b.get_height() / 2, f"{b.get_width():.4f}", va="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.savefig(figures_dir / "06_brier_score_comparison.png", dpi=300)
plt.savefig(figures_dir / "06_brier_score_comparison.svg")
plt.show()

### 6.4. Diễn giải kết quả khách quan & Khoa học
1. **Trong nhóm GNN (Tầng 2)**:
   - **MPNN** đạt hiệu năng vượt trội toàn diện so với các kiến trúc GNN khác: AUC-PR = **0.5747** (so với GCN: 0.3261, GraphSAGE: 0.3763, GATv2: 0.2983).
   - MPNN sử dụng mạng nơ-ron truyền thông điệp dựa trên đặc trưng cạnh (edge features: góc chuyền, chiều dài, độ cao) kết hợp đặc trưng đỉnh, giúp nắm bắt ngữ cảnh đường chuyền hiệu quả hơn GCN thuần túy.
2. **So sánh với Baseline phi đồ thị (Tầng 3)**:
   - Hiện tại, **Random Forest** (AUC-PR = **0.6234**) và **ANN** (AUC-PR = **0.6141**) có AUC-PR cao hơn MPNN (0.5747).
   - **Lý do**: Baseline dạng bảng được trang bị các đặc trưng tổng hợp toàn cục (global aggregated features) như: khoảng cách tới khung thành ở cuối đợt tấn công (`end_dist_to_goal`), độ trực diện (`directness`), tiến triển lãnh thổ (`territory_gained`). Trong bóng đá, vị trí kết thúc possession tương quan rất mạnh với xác suất sút.
   - **Kết luận**: GNN học biểu diễn cấu trúc mạng chuyền bóng (relational passing topology), là nền tảng cốt lõi để thực hiện **Giải thích XAI (Bước 7)** nhằm xác định cầu thủ và đường chuyền có đóng góp quan trọng nhất vào cơ hội ghi bàn.

## 7. Kiểm tra chất lượng (Quality Checks)

In [ ]:
checks = []
checks.append({"Check": "Bảng model_comparison có đủ 7 mô hình", "Status": "PASS" if len(df_models) >= 7 else "FAIL"})
checks.append({"Check": "MPNN là best GNN backbone", "Status": "PASS" if df_models[df_models["tier"].str.contains("GNN")]["auc_pr"].idxmax() == df_models[df_models["model"].str.contains("MPNN")].index[0] else "FAIL"})
checks.append({"Check": "AUC-PR nằm trong khoảng [0, 1]", "Status": "PASS" if ((df_models["auc_pr"] >= 0) & (df_models["auc_pr"] <= 1)).all() else "FAIL"})

df_eval_checks = pd.DataFrame(checks)
display(df_eval_checks)
assert (df_eval_checks["Status"] == "PASS").all(), "Có kiểm tra đánh giá không đạt!"

## 8. Output được lưu ở đâu
- Bảng tổng hợp: `results/tables/official/model_comparison.csv`, `baseline_comparison.csv`.
- Biểu đồ:
  - `results/figures/official/06_model_auc_comparison.png` (và `.svg`)
  - `results/figures/official/06_pr_roc_curves.png` (và `.svg`)
  - `results/figures/official/06_brier_score_comparison.png` (và `.svg`)

## 9. Cách tái chạy
```bash
python scripts/06_train_baselines.py --config configs/data.yaml
python scripts/07_evaluate.py --config configs/data.yaml
```

## 10. Giới hạn và giả định
- **Bước 7–10 (XAI / Hybrid Centrality) CHƯA được triển khai trong task này**: Bước 6 chỉ dừng lại ở việc đánh giá hiệu năng dự đoán (Bài toán 1). MPNN sẽ là mô hình GNN đại diện được chuyển tiếp sang phân tích giải thích ở Bước 7.